In [149]:
import os
from dotenv import load_dotenv
import json
import httpx
from datetime import datetime
from zoneinfo import ZoneInfo
from zai import ZaiClient

load_dotenv('.env')

api_key = os.getenv('API_KEY')

client = ZaiClient(api_key=api_key) # creating client connection

In [150]:
def llm_model(prompt) -> str:
    instructions = """
You are the decision engine of an AI agent.

Your job is to:
1. Understand the user's request.
2. Identify the user's intent.
3. Select the appropriate tool.
4. Extract the structured parameters required by that tool.
5. Return the result in JSON format.

You have three tools available:

1. Schedule Management Tool
2. Location Info Tool
3. Simple Analytics Tool

If the user's request does not match any of these tools, return:
{
    "intent": "unsupported"
}

For Schedule Management Tool:
- Use this tool when the user wants to create, update, or delete an event.
- Extract the action, date, time, and event name.
- Return:
{
    "intent": "schedule_management",
    "action": "create/update/delete",
    "date": "...",
    "time": "...",
    "event_name": "..."
}

For Location Info Tool:
- Use this tool when the user asks for information about a location.
- Extract the location name.
- Return:
{
    "intent": "location_info",
    "location": "..."
}

For Simple Analytics Tool:
- Use this tool when the user asks to perform calculations on a list of numbers or values.
- Extract the numbers or values from the user's request.
- Return:
{
    "intent": "simple_analytics",
    "values": [...]
}

Always return valid JSON.
Do not add explanations or extra text outside the JSON.
"""

    response = client.chat.completions.create(
        model="glm-4.7-flash",
        messages=[
            {
                "role": "system",
                "content": instructions
            },
            {
                "role": "user",
                "content": prompt
            }
        ]
    )
    return response.choices[0].message.content

In [151]:
prompt = input("Enter your task : ").strip()
result = llm_model(prompt)
result = result.replace('```json', '').replace('```', '')
user_input = json.loads(result)

def analytics(numbers):
    try:
        total = 0
        for number in numbers :
            total += number
        average = total / len(numbers)
        numbers.sort()
        maximum = numbers[-1]
        minimum = numbers[0]
        numbers_count = len(numbers)
        return f"Average: {average}, Maximum: {maximum}, Minimum: {minimum}, Count: {numbers_count}"
    except (ValueError, ZeroDivisionError,TypeError):
        return f"Sorry Invalid Input"

def location_tool(location):
    try:
        url = "https://geocoding-api.open-meteo.com/v1/search?name=" + location
        req_res = httpx.get(url).json()
        country = req_res['results'][0]['country']
        timezone = req_res['results'][0]['timezone']
        current_time = datetime.now(ZoneInfo(timezone))

        return f"Country is : {country} , TimeZone is : {timezone} and Current Time is : {current_time}"
    except KeyError:
        return "Invalid Location"

events = [
    {"date": "2026-09-05", "time": "09:00", "event_name": "Backend Study"},
    {"date": "2026-09-05", "time": "10:30", "event_name": "AI Training"},
    {"date": "2026-09-05", "time": "12:00", "event_name": "Lunch"},
    {"date": "2026-09-05", "time": "14:00", "event_name": "Meeting with Ahmed"},
    {"date": "2026-09-05", "time": "16:00", "event_name": "Project Work"},
    {"date": "2026-09-05", "time": "18:30", "event_name": "Gym"}
]


def schedule_tool(date, time, event_name, action):

    if action == 'create':

        for event in events:
            if event['date'] == date and event['time'] == time:
                return "WARNING => CONFLICT"

        events.append({
            "date": date,
            "time": time,
            "event_name": event_name
        })

        return f"{event_name} is scheduled successfully on {date} at {time}"

    elif action == 'update':

        for event in events:
            if event['date'] == date and event['event_name'] == event_name:

                event['time'] = time

                return f"{event_name} has been updated to {date} at {time}"

        return "Event not found"

    elif action == 'delete':

        for event in events:
            if event['date'] == date and event['event_name'] == event_name:

                events.remove(event)

                return f"{event_name} has been deleted successfully"

        return "Event not found"

    else:
        return "Invalid action"
    

def decision():
    if user_input['intent'] == 'simple_analytics':
        return analytics(user_input['values'])
    elif user_input['intent'] == 'location_info':
        return location_tool(user_input['location'])
    else:
        return schedule_tool(user_input['date'],user_input['time'],user_input['event_name'],user_input['action'])

In [152]:
print(decision())

Average: 20.0, Maximum: 35, Minimum: 5, Count: 4
